# Pembersihan Korpus PDF Putusan PN Surabaya

Tujuan: mengubah 273 PDF di `PDF_Putusan/` menjadi teks bersih di `CLEAN_TXT/`
plus satu file CSV ringkas (`_MANIFEST.csv`) yang berisi nomor perkara dan nama hakim.

**Cara pakai:** jalankan cell berurutan dari atas ke bawah (Kernel > Restart & Run All).
Ubah `DRY_RUN = True` di Cell 2 kalau hanya ingin melihat rencana tanpa memindah file.

**Prasyarat:** `poppler-utils` terinstal (`pdftohtml`, `pdfinfo`).
Hanya memakai library standar Python (`re`, `shutil`, `hashlib`, `csv`, `subprocess`).

In [1]:
import csv
import hashlib
import html
import re
import shutil
import subprocess
from pathlib import Path

# ---------- Konfigurasi (satu-satunya bagian yang perlu diubah) ----------

FOLDER_SUMBER = Path("PDF_Putusan")
FOLDER_BERSIH  = Path("CLEAN_TXT")
FOLDER_SISIH   = Path("_EXCLUDE")
FILE_MANIFEST  = Path("_MANIFEST.csv")
DRY_RUN   = False   # True = hanya mencetak rencana, tidak memindah/menulis apa pun
BATAS_FILE = 0      # 0 = proses semua file; isi angka kecil (mis. 5) untuk uji coba

# ---------- Konstanta filter font (hasil audit: berlaku di 273/273 file) ---
UKURAN_WATERMARK  = 40        # watermark "Mahkamah Agung Republik Indonesia" selalu >= 40pt
WARNA_HEADER      = "#3b3b3b"  # header "Direktori Putusan" + URL selalu warna ini
UKURAN_DISCLAIMER = 8         # blok Disclaimer selalu 8pt
TOLERANSI_BARIS   = 4         # dua kotak teks dianggap sebaris bila beda 'top' <= 4 px
BATAS_TANPA_TEKS  = 2000      # file 1 halaman dengan teks < ini dianggap hasil scan
BATAS_ABORT       = 20        # pengaman: batalkan pemindahan bila file bermasalah > ini

In [2]:
# ================= BAGIAN 1: baca PDF menjadi kotak-kotak teks =================

def jalankan_pdftohtml(path_pdf):
    """Jalankan poppler, kembalikan XML sebagai string. Flag -i = jangan tulis gambar."""
    hasil = subprocess.run(
        ["pdftohtml", "-xml", "-stdout", "-hidden", "-i", str(path_pdf)],
        capture_output=True, text=True,
    )
    return hasil.stdout


def baca_fontspec(xmlek):
    """Kembalikan {id_font: (ukuran, keluarga, warna)}."""
    gaya = {}
    pola = r'<fontspec id="(\d+)" size="([\d.]+)" family="([^"]+)" color="([^"]+)"'
    for cocok in re.finditer(pola, xmlek):
        gaya[cocok.group(1)] = (float(cocok.group(2)), cocok.group(3), cocok.group(4))
    return gaya


def baca_kotak_teks(xmlek):
    """Kembalikan daftar kotak: {halaman, atas, kiri, id_font, teks}."""
    kotak = []
    for nomor, halaman in enumerate(re.findall(r"<page.*?</page>", xmlek, re.S)):
        for cocok in re.finditer(r"<text ([^>]*)>(.*?)</text>", halaman, re.S):
            atribut = dict(re.findall(r'(\w+)="([^"]*)"', cocok.group(1)))
            teks = html.unescape(re.sub(r"<[^>]+>", "", cocok.group(2)))
            teks = re.sub(r"\s{2,}", " ", teks).strip()
            if teks:
                kotak.append({"halaman": nomor, "atas": int(atribut["top"]),
                            "kiri": int(atribut["left"]),
                            "id_font": atribut.get("font"), "teks": teks})
    return kotak

def jumlah_halaman(path_pdf):
    """Ambil jumlah halaman lewat pdfinfo (cepat, tanpa mengekstrak teks)."""
    keluar = subprocess.run(["pdfinfo", str(path_pdf)],

                            capture_output=True, text=True).stdout
    cocok = re.search(r"Pages:\s+(\d+)", keluar)
    return int(cocok.group(1)) if cocok else 0

In [3]:
# ================= BAGIAN 2: buang watermark / header / disclaimer ============

# Kuncinya: keputusan berdasar UKURAN dan WARNA font, bukan menebak isi teks.



def ambil_font(kotak, fontspec):
    """Kembalikan (ukuran, keluarga, warna) untuk sebuah kotak teks."""
    return fontspec.get(kotak["id_font"], (0, "", ""))


def saring_kotak(daftar_kotak, fontspec):
    """Buang kotak watermark, header situs, dan disclaimer."""
    bersih = []
    for kotak in daftar_kotak:
        ukuran, _, warna = ambil_font(kotak, fontspec)
        if ukuran >= UKURAN_WATERMARK:
            continue
        if warna == WARNA_HEADER:
            continue
        if ukuran == UKURAN_DISCLAIMER:
            continue
        bersih.append(kotak)
    return bersih

In [4]:
# ================= BAGIAN 3: susun kotak menjadi baris teks ==================
def gabung_jadi_baris(daftar_kotak, toleransi=TOLERANSI_BARIS):
    """Urutkan (halaman, atas, kiri); kotak yang tingginya mirip digabung sebaris."""
    urut = sorted(daftar_kotak, key=lambda k: (k["halaman"], k["atas"], k["kiri"]))
    baris, tampal = [], None
    for kotak in urut:
        kunci = (kotak["halaman"], round(kotak["atas"] / toleransi))
        if tampal and kunci == tampal["kunci"]:
            tampal["teks"] += " " + kotak["teks"]
        else:
            tampal = {"kunci": kunci, "teks": kotak["teks"]}
            baris.append(tampal)
    return [b["teks"] for b in baris]


# Kamus perbaikan kata berhuruf renggang ("P U T U S A N" -> "PUTUSAN").
# Hanya berisi kata yang sudah diverifikasi muncul di korpus ini.
KATA_RENGGANG = {
    "P U T U S A N": "PUTUSAN", "P E N E T A P A N": "PENETAPAN",
    "M E N G A D I L I": "MENGADILI", "M E N E T A P K A N": "MENETAPKAN",
    "M E N U N T U T": "MENUNTUT", "M E L A W A N": "MELAWAN",
    "S U B S I D A I R": "SUBSIDAIR", "K E D U A": "KEDUA",
    "A T A U": "ATAU", "A t a u": "Atau", "M A K A": "MAKA",
    "M a k a": "Maka", "B a h w a": "Bahwa", "N a m a": "Nama",
    "P e k e r j a a n": "Pekerjaan", "A g a m a": "Agama",
    "L a w a n": "Lawan", "L A W A N": "LAWAN", "l a w a n": "lawan",
    "M e l a w a n": "Melawan", "U m u r": "Umur",
    "J u m l a h": "Jumlah", "m e m b a c a": "membaca",
    "m e n d e n g a r": "mendengar", "M e i": "Mei", "J u l i": "Juli",
    "N I H I L": "NIHIL", "A T K": "ATK", "S T M": "STM", "S M K": "SMK",
    "L A W F I R M": "LAW FIRM", "R O C H M A": "ROCHMA",
    "Y U N A W A T": "YUNAWAT", "G U M I N A": "GUMINA",
}

POLA_BARIS_SAMPAH = re.compile(r"^[-\u2013\u2014:.\s]+$")


def gabungkan_huruf_renggang(teks):
    """Ganti kata berhuruf renggang sesuai kamus di atas."""
    for renggang, rapat in KATA_RENGGANG.items():
        teks = teks.replace(renggang, rapat)
    return teks


def rapikan_baris(daftar_baris):
    """Rapatkan spasi, perbaiki 'H a k i m', buang baris isi tanda baca saja."""
    hasil = []
    for baris in daftar_baris:
        baris = re.sub(r"\s{2,}", " ", baris).strip()
        baris = re.sub(r"H\s+a\s+k\s+i\s+m", "Hakim", baris, flags=re.I)
        baris = gabungkan_huruf_renggang(baris)
        if baris and not POLA_BARIS_SAMPAH.match(baris):
            hasil.append(baris)
    return hasil


def sambung_kata_terpotong(daftar_baris):
    """Sambung baris berakhiran tanda hubung dengan baris berikut.
    Tanda hubung SELALU dipertahankan karena di dokumen hukum ia
    hampir selalu bagian dari kata ulang ('Laki-laki', 'masing-masing',
    'perundang-undangan'), bukan pemenggalan kata."""
    hasil = []
    for baris in daftar_baris:
        if hasil and hasil[-1].endswith("-") and baris[:1].islower():
            hasil[-1] = hasil[-1] + baris
        else:
            hasil.append(baris)
    return hasil


def normalisasi_label_nomor(baris):
    """Seragamkan 'No.', 'NOMOR :', 'Perkara Nomor :' menjadi 'Nomor :'."""
    baris = re.sub(r"(?i)^(?:perkara\s+)?n(?:omor|o)\.?\s*:?", "Nomor :", baris, count=1)
    return re.sub(r"^Nomor :(\S)", r"Nomor : \1", baris)


In [5]:
# ================= BAGIAN 4: ambil nama hakim =================================
# Tiga aturan kecil dicoba berurutan; yang pertama berhasil dipakai.
# Menambah aturan baru = tulis satu fungsi + satu baris di DAFTAR_ATURAN.



GELAR = (r"(?:S\.?H\.?|SH\.?|M\.?H\.?|M\.?Hum\.?|Dr\.?|Dra\.?|Drs\.?"
         r"|H\.?J\.?|CPArb\.?|S\.?E\.?|SE\.?|MM\.?|H\.?S\.?|M\.?K\.?)")
POLA_NAMA = re.compile(rf"[A-Z][\w.\'-]*(?:\s+[A-Z][\w.\'-]*)*[,.]?\s*{GELAR}(?:\s*[,.]\s*{GELAR})*")
POLA_KELEBIH = re.compile(r"(?i)diputuskan dalam|ditetapkan dalam|"
                           r"rapat permusyawaratan|oleh\s+kami")
POLA_ANCHOR = re.compile(r"(?i)panitera\s+pengganti\s*[,;]?\s*hakim"
                         r"|hakim\s+tunggal"
                         r"|hakim\s*(?:-?\s*hakim\s*)?(?:ketua|anggota)")
POLA_SISA = re.compile(r"(?i)^(halaman|hal)\b|\bdari\s+\d+\s+(hal|halaman)"
                        r"|^(t\s*\.?\s*t\s*\.?\s*d)")
POLA_TTD = re.compile(r"^(t\s*\.?\s*t\s*\.?\s*d\s*\.?\s*)+$", re.I)
POLA_TUNGGAL_KALIMAT = re.compile(
    r"(?i)(.{0,80}?)\s*,?\s*(?:selaku\s+)?(?:hakim\s+pengadilan\s+negeri"
    r"|hakim\s+ketua\s+majelis)")


def hasil_kosong():
    return {"ketua": None, "anggota": [], "tunggal": None,
            "cara": "tidak-ditemukan"}


def bersihkan_nama(nama):
    """Buang sisa gelar di depan nama (mis. "M.H Teguh" -> "Teguh")."""
    for _ in range(2):
        baru = re.sub(r"^(?:S\.?H\.?|M\.?H\.?|M\.?Hum\.?|Drs?\.?|Dr\.?|H\.?J\.?)\s+",
                      "", nama).strip()
        if baru == nama:
            break
        nama = baru
    return nama


def aturan_oleh_kami(daftar_baris):
    """Pola: "oleh kami, X sebagai Hakim Ketua, Y dan Z masing-masing sebagai Hakim Anggota"."""
    for i, baris in enumerate(daftar_baris):
        if not POLA_KELEBIH.search(baris):
            continue
        kalimat = " ".join(daftar_baris[i:i + 7])
        cocok = re.search(r"(?i)(?:oleh\s+kami\s*[:,]?|oleh,?)\s*(.*?)\s*"
                          r"(?:masing-?\s*masing\s+)?sebagai\s+hakim\s+ketua",
                          kalimat)
        if cocok:
            ketua = (POLA_NAMA.findall(cocok.group(1)) or [None])[0]
            sisa = kalimat[cocok.end():]
            if "masing-masing sebagai" in sisa:
                sisa = sisa.split("masing-masing sebagai")[0]
            anggota = POLA_NAMA.findall(sisa)
            if ketua or anggota:
                return {"ketua": ketua, "anggota": anggota,
                        "tunggal": None, "cara": "oleh-kami"}
        if re.search(r"(?i)hakim\s+tunggal", kalimat):
            nama = POLA_NAMA.findall(kalimat)
            if nama:
                return {"ketua": None, "anggota": [],
                        "tunggal": nama[0], "cara": "oleh-kami-tunggal"}
    return None





def aturan_blok_ttd(daftar_baris):
    """Pola: anchor "Panitera Pengganti, Hakim," lalu baris nama di bawahnya."""
    for i, baris in enumerate(daftar_baris):
        if not POLA_ANCHOR.search(baris):
            continue
        tunggal = bool(re.search(r"(?i)hakim\s+tunggal", baris))
        for berikut in daftar_baris[i + 1:i + 6]:
            if POLA_SISA.search(berikut) or POLA_TTD.match(berikut.strip()):
                continue
            nama = POLA_NAMA.findall(berikut)
            if not nama:
                continue
            kecil = baris.lower()
            if tunggal:
                return {"ketua": None, "anggota": [],
                        "tunggal": nama[0], "cara": "ttd-tunggal"}
            if "ketua" in kecil and "anggota" in kecil:
                if re.search(r"(?i)anggota.*ketua", kecil):
                    ketua = nama[1] if len(nama) > 1 else None
                    return {"ketua": ketua, "anggota": [nama[0]],
                            "tunggal": None, "cara": "ttd-majelis"}
                return {"ketua": nama[0], "anggota": nama[1:],
                        "tunggal": None, "cara": "ttd-majelis"}
            if "panitera" in kecil and "anggota" not in kecil:
                return {"ketua": None, "anggota": [],
                        "tunggal": nama[-1], "cara": "ttd-tunggal"}
            if len(nama) >= 2:
                return {"ketua": nama[0], "anggota": nama[1:],
                        "tunggal": None, "cara": "ttd-majelis"}
            return {"ketua": None, "anggota": [],
                    "tunggal": nama[0], "cara": "ttd-tunggal"}
    return None


def aturan_hakim_kalimat(daftar_baris):
    """Cadangan: "X, S.H., Hakim Pengadilan Negeri Surabaya"."""
    for i, baris in enumerate(daftar_baris):
        kalimat = " ".join(daftar_baris[i:i + 7])
        if POLA_KELEBIH.search(kalimat):
            continue  # sudah ditangani aturan_oleh_kami
        cocok = POLA_TUNGGAL_KALIMAT.search(kalimat)
        if cocok:
            nama = POLA_NAMA.findall(cocok.group(1))
            if nama:
                return {"ketua": None, "anggota": [],
                        "tunggal": nama[-1], "cara": "tunggal-kalimat"}
    return None


DAFTAR_ATURAN = [
    ("oleh-kami", aturan_oleh_kami),
    ("ttd", aturan_blok_ttd),
    ("kalimat", aturan_hakim_kalimat),
]


def ambil_hakim(daftar_baris):
    """Coba setiap aturan berurutan; kembalikan hasil pertama yang berhasil."""
    for _, aturan in DAFTAR_ATURAN:
        hasil = aturan(daftar_baris)
        if hasil:
            if hasil["ketua"]:
                hasil["ketua"] = bersihkan_nama(hasil["ketua"])
            if hasil["tunggal"]:
                hasil["tunggal"] = bersihkan_nama(hasil["tunggal"])
            hasil["anggota"] = [bersihkan_nama(n) for n in hasil["anggota"]]
            return hasil
    return hasil_kosong()


In [6]:
# ================= BAGIAN 5: audit korpus =====================================

# Satu pipeline penuh per file; hasilnya disimpan di CACHE supaya tidak diulang.



def proses_satu_pdf(path_pdf):
    """Pipeline lengkap satu file: baca -> saring -> gabung -> rapikan -> hakim."""
    xmlek = jalankan_pdftohtml(path_pdf)
    fontspec = baca_fontspec(xmlek)
    kotak = baca_kotak_teks(xmlek)
    baris = gabung_jadi_baris(saring_kotak(kotak, fontspec))
    baris = sambung_kata_terpotong(rapikan_baris(baris))
    baris = [normalisasi_label_nomor(b) for b in baris]
    return baris, ambil_hakim(baris)


def sidik_jari(path_pdf):
    """MD5 isi file (untuk menemukan duplikat byte-identik)."""
    return hashlib.md5(Path(path_pdf).read_bytes()).hexdigest()


def audit():
    """Proses semua PDF (sumber + yang sudah disisihkan), kembalikan {nama: info}.
    Menyertakan _EXCLUDE/ supaya manifest tetap 273 baris saat notebook dijalankan ulang."""
    daftar = sorted(p.name for p in FOLDER_SUMBER.glob("*.pdf"))
    daftar += sorted(p.name for p in FOLDER_SISIH.glob("*.pdf"))
    if BATAS_FILE:
        daftar = daftar[:BATAS_FILE]
    info = {}
    for urutan, nama in enumerate(daftar, 1):
        lokasi = FOLDER_SUMBER / nama
        if not lokasi.exists():
            lokasi = FOLDER_SISIH / nama
        baris, hakim = proses_satu_pdf(lokasi)
        teks = "\n".join(baris)
        info[nama] = {
            "md5": sidik_jari(lokasi),
            "halaman": jumlah_halaman(lokasi),
            "baris": baris,
            "karakter": len(teks),
            "hakim": hakim,
        }
        if urutan % 25 == 0:
            print(f"  ... {urutan}/{len(daftar)} file")
    return info

print("Mulai audit korpus...")
CACHE = audit()
print(f"Selesai: {len(CACHE)} file diaudit.")


Mulai audit korpus...
  ... 25/273 file
  ... 50/273 file
  ... 75/273 file
  ... 100/273 file
  ... 125/273 file
  ... 150/273 file
  ... 175/273 file
  ... 200/273 file
  ... 225/273 file
  ... 250/273 file
Selesai: 273 file diaudit.


In [7]:
# ================= BAGIAN 6: tentukan aksi tiap file ==========================
# Aturan eksklusi/rename dibaca dari ISI dokumen, bukan daftar hafalan.

POLA_NOMOR = re.compile(r"(\d+)\s*/?\s*([A-Za-z][\w.\-]*(?:[ /][A-Za-z][\w.\-]*)?)\s*/\s*(\d{4})\s*/\s*P\.?\s*N\.?\s*([\w.\s]*)")
POLA_NAMA_FILE = re.compile(r"^(\d+)_([\w.\-]+?)_(\d{4})_PN", re.I)

def normalisasi_nomor(nomor, jenis, tahun, pengadilan):
    """Seragamkan nomor untuk perbandingan (abaikan spasi/titik/besar-kecil)."""
    bersih = lambda s: re.sub(r"[^A-Z0-9]+", "", s.upper())
    return bersih(nomor), bersih(jenis), tahun, bersih(pengadilan)

def pecah_nama_file(nama):
    """'1031_Pdt.G_2026_PN Sby.pdf' -> ('1031', 'Pdt.G', '2026')."""
    cocok = POLA_NAMA_FILE.search(nama)
    if not cocok:
        return None
    return cocok.group(1), cocok.group(2), cocok.group(3)

def cari_nomor_header(daftar_baris):
    """Cari nomor perkara resmi di 12 baris pertama (blok judul dokumen).
    Prioritas: baris berlabel 'Nomor :' (bukan 'Jo Nomor' atau 'Register Nomor')."""
    awal = daftar_baris[:12]
    for baris in awal:
        if baris.startswith("Nomor :"):
            cocok = POLA_NOMOR.search(baris)
            if cocok:
                return (cocok.group(1), cocok.group(2), cocok.group(3), cocok.group(4))
    for baris in awal:
        cocok = POLA_NOMOR.search(baris)
        if cocok:
            return (cocok.group(1), cocok.group(2), cocok.group(3), cocok.group(4))
    return None

def cari_nomor_footer(daftar_baris):
    """Cari 'Penetapan/Putusan Nomor ...' di bagian footer halaman."""
    for baris in daftar_baris[-15:]:
        if re.search(r"(?i)(penetapan|putusan).*nomor", baris):
            cocok = POLA_NOMOR.search(baris)
            if cocok:
                return (cocok.group(1), cocok.group(2), cocok.group(3), cocok.group(4))
    return None

def tentukan_aksi(nama, info, peta_md5):
    """Kembalikan (aksi, alasan, nama_baru). Aksi: 'normal' | 'sisihkan' | 'rename'."""
    if peta_md5[info["md5"]] != nama:
        return ("sisihkan", f"duplikat byte-identik dari {peta_md5[info['md5']]}", None)
    if info["halaman"] <= 1 and info["karakter"] < BATAS_TANPA_TEKS:
        return ("sisihkan", "hasil scan tanpa layer teks (butuh OCR)", None)
    pecah = pecah_nama_file(nama)
    header = cari_nomor_header(info["baris"])
    
    if pecah and header:
        nomor_f, jenis_f, tahun_f = normalisasi_nomor(*pecah, "")[:3]
        nomor_h, jenis_h, tahun_h, adil_h = normalisasi_nomor(*header)
        if nomor_f != nomor_h:
            return ("sisihkan", f"nomor nama file ({nomor_f}) != nomor isi ({nomor_h})", None)
        if "SBY" not in adil_h and adil_h:
            return ("sisihkan", f"putusan pengadilan lain ({adil_h})", None)
        if tahun_f != tahun_h:
            baru = nama.replace(f"_{tahun_f}_", f"_{tahun_h}_", 1)
            return ("rename", f"tahun nama file ({tahun_f}) -> tahun isi ({tahun_h})", baru)
        if jenis_f != jenis_h:
            return ("normal", f"flag: jenis nama file ({jenis_f}) != jenis isi ({jenis_h})", None)
            
    if pecah and not header:
        return ("normal", "flag: nomor tidak terbaca di header", None)
        
    return ("normal", "", None)

def susun_rencana(info):
    """Hitung aksi semua file + peta md5 (nama kanonis = bukan 'putusan_*')."""
    peta_md5 = {}
    for nama in sorted(info):
        md5 = info[nama]["md5"]
        if md5 not in peta_md5:
            peta_md5[md5] = nama
        elif (peta_md5[md5].lower().startswith("putusan_") and not nama.lower().startswith("putusan_")):
            peta_md5[md5] = nama

    rencana = {nama: tentukan_aksi(nama, info[nama], peta_md5) for nama in info}
    return rencana

RENCANA = susun_rencana(CACHE)
print(f"{'file':38s} {'aksi':10s} alasan")
for nama in sorted(RENCANA):
    aksi, alasan, baru = RENCANA[nama]
    if aksi != "normal" or alasan:
        ekstra = f" -> {baru}" if baru else ""
        print(f"{nama:38s} {aksi:10s} {alasan}{ekstra}")

n_bermasalah = sum(1 for v in RENCANA.values() if v[0] != "normal")
print(f"\nFile bermasalah: {n_bermasalah} dari {len(RENCANA)}")

file                                   aksi       alasan
1031_Pdt.G_2026_PN Sby.pdf             sisihkan   nomor nama file (1031) != nomor isi (100)
17_Pdt.Sus-Pailit-GLL_2025_PN.Niaga Sby.pdf normal     flag: jenis nama file (PDTSUSPAILITGLL) != jenis isi (PDTSUSGUGATANLAINLAIN)
3162_Pid.B_2017_PN SBY.pdf             sisihkan   nomor nama file (3162) != nomor isi (3150)
3749_Pid.B_2017_PN SBY.pdf             sisihkan   nomor nama file (3749) != nomor isi (113)
3882_Pid.Sus_2017_PN SBY.pdf           normal     flag: jenis nama file (PIDSUS) != jenis isi (PIDB)
527_Pid.Sus_2018_PN SBY.pdf            sisihkan   nomor nama file (527) != nomor isi (359)
748_Pid.C_2025_PN Sby.pdf              sisihkan   hasil scan tanpa layer teks (butuh OCR)
946_Pdt.G_2025_PN Sby.pdf              normal     flag: nomor tidak terbaca di header
putusan_186_pdt.g_2026_pn_sby_20260930172538.pdf sisihkan   duplikat byte-identik dari 186_Pdt.G_2026_PN Sby.pdf
putusan_227_pdt.g_2026_pn_sby_20261001175526.pdf sisi

In [8]:
# ================= BAGIAN 7: jalankan aksi + tulis TXT =======================

def pindahkan_ke_exclude(nama, alasan):
    """Pindahkan PDF ke _EXCLUDE/ (idempoten: lewati bila sudah pindah)."""
    asal, tujuan = FOLDER_SUMBER / nama, FOLDER_SISIH / nama
    if tujuan.exists():
        return "sudah-disana"
    if DRY_RUN:
        return "dry-run"
    FOLDER_SISIH.mkdir(exist_ok=True)
    shutil.move(str(asal), str(tujuan))
    return "dipindah"


def rename_file(nama, nama_baru):
    """Ubah nama PDF (idempoten, batal bila target sudah ada)."""
    asal, tujuan = FOLDER_SUMBER / nama, FOLDER_SUMBER / nama_baru
    if not asal.exists():
        return "sudah-rename"
    if tujuan.exists():
        return "batal-ada-kembar"
    if DRY_RUN:
        return "dry-run"
    asal.rename(tujuan)
    return "direname"


def tulis_txt(nama, daftar_baris):
    """Tulis satu file teks bersih (1 baris = 1 unit rule)."""
    FOLDER_BERSIH.mkdir(exist_ok=True)
    tujuan = FOLDER_BERSIH / ("T-" + Path(nama).stem + ".txt")
    if DRY_RUN:
        return str(tujuan)
    tujuan.write_text("\n".join(daftar_baris) + "\n", encoding="utf-8")
    return str(tujuan)


def jalankan_aksi(info, rencana):
    """Eksekusi rencana. Berhenti bila file bermasalah melebihi BATAS_ABORT."""
    bermasalah = sum(1 for v in rencana.values() if v[0] != "normal")
    if bermasalah > BATAS_ABORT:
        raise SystemExit(f"BATAL: {bermasalah} file bermasalah melebihi batas {BATAS_ABORT}.")
    hasil = {}
    for nama in sorted(rencana):
        aksi, alasan, baru = rencana[nama]
        if aksi == "sisihkan":
            hasil[nama] = ("exclude", pindahkan_ke_exclude(nama, alasan), alasan)
        elif aksi == "rename":
            status_rename = rename_file(nama, baru)
            tujuan_txt = tulis_txt(baru, info[nama]["baris"])
            hasil[nama] = (baru, f"{status_rename} + {tujuan_txt}", alasan)
        else:
            tujuan_txt = tulis_txt(nama, info[nama]["baris"])
            hasil[nama] = (nama, tujuan_txt, alasan)
    return hasil





HASIL = jalankan_aksi(CACHE, RENCANA)
n_aktif = sum(1 for nama, (akhir, _, _) in HASIL.items() if not str(akhir).startswith("exclude"))
print(f"Selesai. File aktif: {n_aktif}, disisihkan: {len(HASIL) - n_aktif}.")


Selesai. File aktif: 258, disisihkan: 15.


In [9]:
# ================= BAGIAN 8: tulis manifest CSV ===============================

KOLOM = ["file_pdf", "nama_asli", "status", "alasan", "nomor_header",
         "nomor_footer", "nomor_cocok", "tahun", "jenis", "pengadilan",
         "hakim_ketua", "hakim_anggota", "hakim_tunggal", "cara_hakim",
         "halaman_pdf", "jumlah_baris", "jumlah_karakter"]


def teks_nomor(pecah):
    """('100','Pdt.G','2026','Sby') -> '100/Pdt.G/2026'."""
    if not pecah:
        return ""
    nomor, jenis, tahun = pecah[0], pecah[1], pecah[2]
    return f"{nomor}/{jenis}/{tahun}"


def susun_baris_manifest(nama, info, rencana):
    """Susun satu baris manifest untuk satu file."""
    aksi, alasan, baru = rencana[nama]
    akhir = baru if aksi == "rename" else nama
    status = {"sisihkan": "exclude", "rename": "rename", "normal": "aktif"}[aksi]
    header = cari_nomor_header(info["baris"])
    footer = cari_nomor_footer(info["baris"])
    cocok = ""  # bandingkan nomor+jenis+tahun saja;
    # ekor kalimat di footer ("tertanggal ...") diabaikan
    if header and footer:
        cocok = "ya" if normalisasi_nomor(*header)[:3] \
                == normalisasi_nomor(*footer)[:3] else "tidak"
    hakim = info["hakim"]
    if re.search(r"-{3,}.*S\\.?H", " ".join(info["baris"][-30:])):
        alasan = (alasan + "; " if alasan else "") + "flag: nama hakim tersensor"
    return {
        "file_pdf": akhir, "nama_asli": nama, "status": status, "alasan": alasan,
        "nomor_header": teks_nomor(header), "nomor_footer": teks_nomor(footer),
        "nomor_cocok": cocok,
        "tahun": header[2] if header else "", "jenis": header[1] if header else "",
        "pengadilan": (header[3].strip() if header else ""),
        "hakim_ketua": hakim["ketua"] or "",
        "hakim_anggota": "; ".join(hakim["anggota"]),
        "hakim_tunggal": hakim["tunggal"] or "", "cara_hakim": hakim["cara"],
        "halaman_pdf": info["halaman"], "jumlah_baris": len(info["baris"]),
        "jumlah_karakter": info["karakter"],

    }


def tulis_manifest(info, rencana, hasil):
    """Tulis _MANIFEST.csv (273 baris: aktif + rename + exclude)."""
    baris_data = [susun_baris_manifest(nama, info[nama], rencana) for nama in sorted(info)]
    if DRY_RUN:
        return baris_data
    with open(FILE_MANIFEST, "w", newline="", encoding="utf-8") as berkas:
        penulis = csv.DictWriter(berkas, fieldnames=KOLOM)
        penulis.writeheader()
        penulis.writerows(baris_data)
    alasan = [f"{b['nama_asli']}: {b['alasan']}" for b in baris_data if b["status"] == "exclude"]
    (FOLDER_SISIH / "_ALASAN.txt").write_text("\n".join(alasan) + "\n", encoding="utf-8")
    return baris_data

DATA = tulis_manifest(CACHE, RENCANA, HASIL)
print(f"Manifest ditulis: {len(DATA)} baris -> {FILE_MANIFEST}")


Manifest ditulis: 273 baris -> _MANIFEST.csv


In [10]:
# ================= BAGIAN 9: validasi =========================================
def hitung_trigger(daftar_baris, pola):
    """Hitung baris yang cocok dengan satu pola regex."""
    return sum(1 for b in daftar_baris if re.search(pola, b, re.I))


TRIGGER = {
    "nama (lengkap|terdakwa) :": r"nama (lengkap|terdakwa) :",
    "menyatakan terdakwa": r"menyatakan terdakwa",
    "menjatuhkan pidana": r"menjatuhkan pidana",
    "diputuskan dalam": r"diputuskan dalam",
    "hakim ketua": r"hakim ketua",
}

NOISE = [r"^Mahkamah Agung Republik Indonesia$", r"mahkamahagung\.go\.id",
         r"^Disclaimer$", r"^Direktori Putusan Mahkamah Agung Republik Indonesia$"]


def validasi(info, rencana):
    """Cetak rekap: file, karakter, sisa noise, cakupan trigger, sampel teks."""
    aktif = [n for n, v in rencana.items() if v[0] != "sisihkan"]
    total_karakter = sum(info[n]["karakter"] for n in aktif)
    print(f"File aktif : {len(aktif)}")
    print(f"File sisihan: {len(info) - len(aktif)}")
    print(f"Total karakter bersih: {total_karakter:,}")
    print("\nSisa noise di CLEAN_TXT/ (harus 0 semua):")
    for pola in NOISE:
        kena = sum(hitung_trigger(info[n]["baris"], pola) for n in aktif)
        print(f"  {kena:6d}  {pola}")
    print("\nCakupan trigger rule (jumlah file yang cocok):")
    for nama_trigger, pola in TRIGGER.items():

        kena = sum(1 for n in aktif if hitung_trigger(info[n]["baris"], pola))

        print(f"  {kena:4d}/{len(aktif)}  {nama_trigger}")

    print("\nFile tanpa hakim:")

    for baris_manifest in DATA:

        if baris_manifest["status"] == "aktif" and not (

                baris_manifest["hakim_ketua"] or baris_manifest["hakim_anggota"]

                or baris_manifest["hakim_tunggal"]):

            print(f"   {baris_manifest['file_pdf']}")

    print("\nSampel 30 baris pertama (1 pidana, 1 gugatan, 1 permohonan):")

    for contoh in ["1002_Pid.Sus-LH_2026_PN Sby.pdf", "108_Pdt.G_2026_PN Sby.pdf",

                   "2296_Pdt.P_2026_PN Sby.pdf"]:

        if contoh in info and rencana[contoh][0] != "sisihkan":

            print(f"\n===== {contoh} =====")

            for baris in info[contoh]["baris"][:30]:

                print(" ", baris[:100])





validasi(CACHE, RENCANA)


File aktif : 258
File sisihan: 15
Total karakter bersih: 11,069,896

Sisa noise di CLEAN_TXT/ (harus 0 semua):
       0  ^Mahkamah Agung Republik Indonesia$
       0  mahkamahagung\.go\.id
       0  ^Disclaimer$
       0  ^Direktori Putusan Mahkamah Agung Republik Indonesia$

Cakupan trigger rule (jumlah file yang cocok):
    95/258  nama (lengkap|terdakwa) :
    99/258  menyatakan terdakwa
   100/258  menjatuhkan pidana
   140/258  diputuskan dalam
   160/258  hakim ketua

File tanpa hakim:

Sampel 30 baris pertama (1 pidana, 1 gugatan, 1 permohonan):

===== 1002_Pid.Sus-LH_2026_PN Sby.pdf =====
  PUTUSAN
  Nomor : 1002/Pid.Sus-LH/2026/PN Sby
  DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
  Pengadilan Negeri Surabaya yang mengadili perkara pidana dengan
  acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
  berikut dalam perkara Terdakwa :
  1. Nama lengkap :
  HENDRIK SETIAWAN BIN (alm) ABDUL
  HALIM;
  2. Tempat lahir : Jember;
  3. Umur/tanggal lahir : 3

## Ringkasan hasil

Cell 9 mencetak semuanya: jumlah file aktif vs disisihkan, sisa noise (harus nol),
cakupan tiap trigger rule, daftar file tanpa hakim, dan sampel teks untuk inspeksi manual.

File yang perlu perhatian manusia ada di kolom `alasan` pada `_MANIFEST.csv`
(mis. `flag: ...`). Khusus kolom hakim: cek 20 baris dengan `cara_hakim` =
`tunggal-kalimat` atau `ttd-tunggal` terhadap blok tanda tangan di PDF aslinya.

# BAGIAN 10 — EKSTRAKSI 15 ENTITAS DARI CLEAN_TXT
Tujuan: melanjutkan `clean_pdf.ipynb` menjadi satu file pipeline.
- Input: `CLEAN_TXT/` (258 file .txt, prefix `T-`)
- Output: `OUTPUT_CLEAN/` (per file `O-*.txt` berisi entitas terstruktur) + `OUTPUT_CLEAN/entitas_258.csv` (rangkuman 258 data)
- Desain: fungsi-fungsi kecil (rule-based), regex terpisah, komentar Bahasa Indonesia.
- Nilai kosong memakai `-`; kolom keterangan menjelaskan alasan setiap kekosongan.



In [11]:
# ================= BAGIAN 10: Konfigurasi Ekstraksi Entitas =================
from pathlib import Path
import re
import csv
from collections import Counter

INPUT_TXT = Path("CLEAN_TXT")
OUTPUT_CLEAN = Path("OUTPUT_CLEAN")
OUTPUT_CLEAN.mkdir(exist_ok=True)
KOLOM_ENTITAS = ["nomor_putusan","nama_terdakwa_1","nama_terdakwa_2","nama_terdakwa_3","tuntutan_pidana","tuntutan_kuhp","tuntutan_hukuman","putusan_pidana","putusan_hukuman","tanggal_putusan","hakim_ketua","hakim_anggota_1","hakim_anggota_2","panitera","penuntut_umum"]
KOLOM_TAMBAH = ["file_txt","jenis_dokumen","entitas_berlaku","total_entitas_terisi","status_lengkap","catatan_kosong"]
KOLOM_CSV = KOLOM_ENTITAS + KOLOM_TAMBAH
TANDA_KOSONG = "-"
BATAS_PREVIEW = 5
CONTOH_DEMO = "T-1016_Pid.B_2026_PN Sby.txt"
LABEL_CETAK = {"nomor_putusan":"Nomor Putusan","nama_terdakwa_1":"Nama Terdakwa 1","nama_terdakwa_2":"Nama Terdakwa 2","nama_terdakwa_3":"Nama Terdakwa 3","tuntutan_pidana":"Tuntutan Pidana","tuntutan_kuhp":"Tuntutan KUHP","tuntutan_hukuman":"Tuntutan Hukuman","putusan_pidana":"Putusan Pidana","putusan_hukuman":"Putusan Hukuman","tanggal_putusan":"Tanggal Putusan","hakim_ketua":"Hakim Ketua","hakim_anggota_1":"Hakim Anggota 1","hakim_anggota_2":"Hakim Anggota 2","panitera":"Panitera","penuntut_umum":"Penuntut Umum"}
JEJAK_ATURAN = {"nomor_putusan":"baris judul 'Nomor :' + POLA_NOMOR","nama_terdakwa_1":"label Nama di blok identitas pidana","nama_terdakwa_2":"label Nama kedua di blok identitas pidana","nama_terdakwa_3":"label Nama ketiga di blok identitas pidana","tuntutan_pidana":"amar 1 Tuntutan + frasa sesudah 'bersalah melakukan'","tuntutan_kuhp":"amar 1 Tuntutan + pola 'Pasal ... KUHP'","tuntutan_hukuman":"amar 2 Tuntutan + durasi sesudah 'selama'","putusan_pidana":"amar 1 MENGADILI + frasa sesudah 'bersalah melakukan'","putusan_hukuman":"amar 2 MENGADILI + durasi sesudah 'selama'","tanggal_putusan":"kalimat 'Demikian ...' + pola 'pada hari, tanggal'","hakim_ketua":"nama sebelum 'sebagai/selaku Hakim Ketua'","hakim_anggota_1":"nama pertama sebelum 'masing-masing ... Anggota'","hakim_anggota_2":"nama kedua sebelum 'masing-masing ... Anggota'","panitera":"nama sesudah 'dibantu oleh' sebelum 'Panitera Pengganti'","penuntut_umum":"nama sesudah 'dihadiri oleh' sebelum 'Penuntut Umum'"}


In [12]:
# ================= BAGIAN 10: Konstanta & Regex =================
POLA_BARIS_NOMOR = re.compile(r"(?i)^\s*(?:[IVX]+\.\s*)?(?:putusan\s+|penetapan\s+)?nomor\b")
POLA_NOMOR = re.compile(r"(?i)(\d{1,5}|-{5,})\s*/?\s*([A-Za-z][\w.\-]*(?:[ /][A-Za-z][\w.\-]*)*?)\s*/\s*(\d{4})(?:\s*/\s*P\s*\.?\s*N\s*\.?\s*((?:Niaga\s+)?Sby|SBY))?")
POLA_LABEL_NAMA = re.compile(r"(?i)(?:^|\b)(?:[IVX0-9]+\.\s*)?(?:nama\s+(?:lengkap|terdakwa)|namalengkap|nama)\s*:")
POLA_BATAS_IDENTITAS = re.compile(r"(?i)terdakwa\s+dita(?:han|ngkap)|telah\s+ditahan|terdakwa\s+berada|ditahan\s*/\s*ditangkap|dibantu\s+(?:oleh\s+)?(?:penasihat|penasehat|advokat|para)|pengadilan negeri tersebut")
POLA_LABEL_LAPANGAN = re.compile(r"(?i)^\s*(?:[IVX0-9]+\.\s*)?(?:tempat(?:\s+tanggal)?(?:\s+lahir)?|umur|jenis|kebangsaan|kewarganegaraan|agama|pekerjaan|pendidikan|alamat|tanggal\s+lahir)\b[^:\n]*:")
POLA_AWAL_TUNTUTAN = re.compile(r"(?i)tuntutan\s+(?:pidana|jaksa|penuntut)|pembacaan\s+tuntutan|tuntutan\s+dari|yang\s+pada\s+pokoknya\s+menuntut")
POLA_AKHIR_TUNTUTAN = re.compile(r"(?i)Setelah mendengar (?:pembelaan|permohonan|tanggapan)|mengajukan pembelaan|tanggapan .*tuntutan|Menimbang,\s*bahwa\s+Terdakwa\s+didakwa|Terdakwa diajukan.*dakwaan|dengan dakwaan sebagai berikut|^\s*PERTAMA\s*$")
POLA_MENGADILI = re.compile(r"(?i)^\s*(?:\d+\.\s*)?M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*$")
POLA_PENUTUP = re.compile(r"(?i)Demikian(?:lah)?[,\s]+(?:(?:putusan|penetapan)(?: ini| tersebut| itu)?\s+)?(?:diambil|diputus(?:kan)?|ditetapkan|diucapkan)\b")
POLA_NOMOR_AMAR = re.compile(r"^\s*(\d+)\s*[.)]\s*(.*)$")
POLA_GARIS_AMAR = re.compile(r"(?i)^\s*[-•]\s+(?=.*\b(menyatakan|menjatuhkan|menetapkan|menghukum|membebankan|memerintahkan)\b)")
def awal_amar_baru(baris):
    """True jika baris memulai amar baru (nomor atau garis ber kata kerja amar)."""
    return bool(POLA_NOMOR_AMAR.match(baris) or POLA_GARIS_AMAR.match(baris))
POLA_BARIS_SAMPAH = re.compile(r"(?i)^\s*(?:Halaman\b|Hal\.?\s*\d+|Putusan\s+No\b|Penetapan\s+No\b|Putusan Nomor\b|Penetapan Nomor\b|\d+)\s*$")
POLA_SAMPAH_HALAMAN = re.compile(r"(?i)\bHal(?:aman)?\.?\s*\d+|\b\d+\s*dari\s*\d+\s*hal\b")
def baris_sampah_amar(x):
    """Deteksi footer halaman ('Halaman 8', 'Hal. 26 dari 27 hal. Put. Nomor ...')."""
    rendah = x.lower()
    return bool(POLA_BARIS_SAMPAH.match(x) or POLA_SAMPAH_HALAMAN.search(x) or ("halaman" in rendah and ("putusan" in rendah or "penetapan" in rendah or re.match(r"^\s*hal", rendah))))
FRAGMEN_NAMA = r"[A-ZÀ-Ž][\w.'’“”`\-]*"
TOKEN_GELAR = r"(?:[A-Z][\w]*\.(?:[A-Z\w.]*\.?)?|S\.?H\.?|M\.?H\.?|M\.?Hum\.?(?:um)?|MKn\.?(?:H)?|Drs?\.?|Dr\.?|SH|MH|S\.?E\.?|SE|S\.?Si\.?(?:Hum)?|S\.?Pd\.?|S\.?Kom\.?(?:Hum)?|S\.?Sos\.?(?:Hum)?|S\.?T\.?(?:Hum)?|A\.?Md\.?(?:Kep)?|H\.?J\.?(?:Hum)?|S\.Hut\.?)"
PEMISAH_GELAR = fr"(?:[\s,]*,[\s,]*|\s+(?={TOKEN_GELAR}\b))"
RANTAI_GELAR = fr"(?:(?:[\s,]*,[\s,]*|\s+(?={TOKEN_GELAR}\b)){TOKEN_GELAR})*"
TEKS_NAMA = fr"{FRAGMEN_NAMA}(?:\s+{FRAGMEN_NAMA})*?{PEMISAH_GELAR}{TOKEN_GELAR}{RANTAI_GELAR}"
POLA_NAMA_BERGELAR = re.compile(fr"\b({TEKS_NAMA})")
POLA_PANEL_HAKIM = re.compile(r"(?s)(?i:oleh\s+kami|oleh)\s*[,:;]?\s*" fr"(?P<ketua>{TEKS_NAMA})\s*,?\s*(?i:sebagai|selaku)\s*(?:(?i:Hakim)\s*(?i:Ketua)(?:\s+Majelis)?|(?i:Ketua)\s*(?i:Majelis)(?:\s+(?i:Hakim))?)" r"\s*,?\s*(?P<anggota>.+?)\s+(?i:masing)[\s-]*(?i:masing)\s+(?i:sebagai|selaku)\s+(?i:Hakim)(?:-(?i:Hakim))?\s+(?i:Anggota)" r"(?:\s+Majelis)?")
POLA_PANEL_HAKIM_ADHOC = re.compile(r"(?s)(?i:oleh\s+kami|oleh)\s*[,:;]?\s*" fr"(?P<ketua>{TEKS_NAMA})\s*,?\s*(?:(?i:sebagai|selaku)\s+)?(?:(?i:Hakim)\s*(?i:Ketua)(?:\s+Majelis)?|(?i:Ketua)\s*(?i:Majelis)(?:\s+(?i:Hakim))?)" r"\s*,?\s*(?P<anggota>.+?)\s+(?i:sebagai|selaku)\s+(?i:Hakim)(?:-(?i:Hakim))?\s+(?i:Anggota)")
POLA_PANEL_TANPA_LABEL = re.compile(r"(?s)(?i:oleh\s+kami|oleh)\s*[,:;]?\s*" fr"(?P<ketua>{TEKS_NAMA})\s*,?\s*(?:(?i:sebagai|selaku)\s+)?(?:(?i:Hakim)\s*(?i:Ketua)(?:\s+Majelis)?|(?i:Ketua)\s*(?i:Majelis)(?:\s+(?i:Hakim))?)" r"\s*,\s*(?P<anggota>.+?)\s*,\s*(?:(?i:penetapan)|(?i:putusan))\s+(?i:tersebut)")
POLA_PANITERA = re.compile(fr"(?s)(?i:didampingi|dibantu|dihadiri)(?:(?i:\s+oleh))?\s*:?\s*(?P<nama>{TEKS_NAMA})(?i:.*?Panitera\s+Pengganti)")
POLA_JPU = re.compile(fr"(?s)(?i:dihadiri|hadiri|dihadapan)(?:(?i:\s+pula))?(?:(?i:\s+oleh))?\s*:?\s*(?P<nama>{TEKS_NAMA})(?i:.*?(?:Jaksa\s+)?Penuntut\s+Umum)")
POLA_JPU_POLOS = re.compile(fr"(?s)(?i:dihadiri|hadiri|dihadapan)(?:(?i:\s+pula))?(?:(?i:\s+oleh))?\s*:?\s*(?P<nama>[A-ZÀ-Ž][\w.'’\-]*(?:\s+[A-ZÀ-Ža-z][\w.'’\-]*){{0,6}}?)\s*,?\s*(?:(?i:Jaksa)\s+)?(?i:Penuntut)\s+(?i:Umum)")
POLA_TANGGAL = re.compile(r"(?i)pada\s+(?:hari\s*:?\s*)?((?:[A-Za-z’]\s*){2,12}?)\s*,?\s*(?:tanggal\s*:?\s*|tangga\s*:?\s*|tangal\s*:?\s*)?(\d{1,2})\s+((?:januari|februari|pemberi|pebruari|maret|april|mei|juni|juli|agustus|september|sepetember|oktober|nopember|november|desember))\s+(\d{4})")
POLA_HUKUMAN_KHUSUS = re.compile(r"(?i)\bpidana mati\b|\bhukuman mati\b|\bpenjara seumur hidup\b|\bseumur hidup\b")
SATUAN_WAKTU = {"tahun", "bulan", "minggu", "hari"}
KATA_BERHENTI_LAMA = {"dikurangi", "dikurangkan", "dipotong", "potong", "dengan", "apabila", "yang", "untuk", "agar", "supaya", "serta", "denda", "pidana", "penjara", "tetap", "ditahan", "masa", "tahanan"}


In [ ]:
# ================= BAGIAN 10: Fungsi bantu dasar =================
def baca_baris(path: Path):
    """Baca .txt menjadi baris asli dan baris huruf kecil."""
    try:
        teks = path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        teks = path.read_text(encoding="latin-1", errors="replace")
    baris = teks.splitlines()
    return baris, [x.lower() for x in baris]
def rapikan_teks(teks):
    """Rapikan spasi; buang ; , : di ujung, tetapi pertahankan titik gelar."""
    if not teks:
        return None
    teks = re.sub(r"\s+", " ", teks).strip().lstrip(";:,. ").rstrip(";, :").strip()
    if len(teks) >= 2 and teks[0] in "\"'“‘" and teks[-1] in "\"'”’":
        teks = teks[1:-1].strip()  # buang tanda kutip pembungkus luar
    elif teks[:1] in "'‘":
        teks = teks[1:].strip()  # kutip pembuka tanpa penutup
    return teks or None
def buang_baris_sampah(daftar):
    """Buang footer halaman dan nomor halaman yang tersisa."""
    return [x for x in daftar if x.strip() and not baris_sampah_amar(x)]
def jendela(baris, i, sebelum=5, sesudah=8):
    """Gabungkan beberapa baris sebelum/sesudah menjadi satu kalimat."""
    awal = max(0, i - sebelum)
    akhir = min(len(baris), i + sesudah)
    return " ".join(baris[awal:akhir])
def cari_di_jendela(baris_low, baris_asli, pemicu, pola):
    """Cari pemicu per baris, lalu terapkan pola pada jendelanya."""
    for i, x in enumerate(baris_low):
        if pemicu in x:
            cocok = pola.search(jendela(baris_asli, i))
            if cocok:
                return cocok
    return None
def bersihkan_nama_terdakwa(teks):
    """Ambil nama sebelum ';' dan buang keterangan lahir/alamat."""
    if not teks:
        return None
    depan = re.split(r";", teks, maxsplit=1)[0]
    depan = re.split(r",\s*(?:lahir|bertempat|selanjutnya|disebut|pekerjaan|alamat|jenis|agama|kebangsaan|umur|tempat|status|kewarganegaraan)\b", depan, maxsplit=1, flags=re.I)[0]
    return rapikan_teks(depan)
def pecah_nama_bergelar(teks):
    """Pecah deretan nama bergelar ('X, S.H. dan Y, S.H.')."""
    hasil = []
    if not teks:
        return hasil
    for nama in POLA_NAMA_BERGELAR.findall(teks):
        bersih = rapikan_nama(nama)
        if bersih and len(bersih) > 3 and bersih not in hasil:
            hasil.append(bersih)
    return hasil
def rapikan_nama(nama):
    """Rapikan nama: buang koma ganda dan beri spasi sesudah koma."""
    nama = rapikan_teks(nama)
    if not nama:
        return None
    nama = re.sub(r"(?<=[a-zà-ž])\.,(?=[A-Z])", ",", nama)
    return re.sub(r",(?=[^\s\d])", ", ", nama)
def ambil_jenis_perkara(nama_file):
    """Ambil token jenis dari nama file T-<nomor>_<jenis>_<tahun>..."""
    inti = Path(nama_file).stem
    if inti.startswith("T-"):
        inti = inti[2:]
    bagian = inti.split("_", 2)
    return bagian[1] if len(bagian) > 1 else ""
    
PETA_JENIS = {"Pid.B":"pidana","Pid.Sus":"pidana","Pid.Sus-LH":"pidana","Pid.Sus-TPK":"pidana","Pid.Pra":"praperadilan","Pdt.P":"permohonan","Pdt.G":"gugatan","Pdt.G.S":"gugatan-sederhana","Pdt.Bth":"bantahan","Pdt.Sus-PHI":"phi","Pdt.Sus-HKI_Merek":"hki-merek","Pdt.Sus-HKI":"hki-merek","Pdt.Sus-PKPU":"pailit-pkpu","Pdt.Sus-Pailit-GLL":"pailit-pkpu","Pdt.Sus-Renvoi Prosedur":"renvoi"}
def nama_jenis_dokumen(nama_file):
    """Ubah token jenis file menjadi label yang mudah dibaca."""
    return PETA_JENIS.get(ambil_jenis_perkara(nama_file), "lainnya")
BERLAKU_PIDANA = {"nomor_putusan","nama_terdakwa_1","nama_terdakwa_2","nama_terdakwa_3","tuntutan_pidana","tuntutan_kuhp","tuntutan_hukuman","putusan_pidana","putusan_hukuman","tanggal_putusan","hakim_ketua","hakim_anggota_1","hakim_anggota_2","panitera","penuntut_umum"}
BERLAKU_UMUM = {"nomor_putusan","tanggal_putusan","hakim_ketua","hakim_anggota_1","hakim_anggota_2","panitera"}
def himpunan_berlaku(jenis):
    """Pidana memakai 15 entitas; selain itu hanya 6 entitas umum."""
    return set(BERLAKU_PIDANA) if jenis == "pidana" else set(BERLAKU_UMUM)


In [14]:
# ================= BAGIAN 10: Segmentasi bagian dokumen =================
def batas_identitas(baris):
    """Identitas terdakwa ada di awal, sebelum status penahanan/pendampingan."""
    for i, x in enumerate(baris[:200]):
        if POLA_BATAS_IDENTITAS.search(x):
            return i
    return min(len(baris), 100)
def batas_tuntutan(baris, baris_low):
    """Tuntutan dimulai dari pembacaan tuntutan dan berakhir sebelum pembelaan/dakwaan."""
    for i, x in enumerate(baris_low):
        if POLA_AWAL_TUNTUTAN.search(x):
            sekitar = " ".join(baris_low[max(0, i - 2):i + 8])
            if "penuntut umum" not in sekitar and "jaksa" not in sekitar and "menuntut" not in sekitar:
                continue
            for j in range(i + 1, min(len(baris), i + 400)):
                if POLA_AKHIR_TUNTUTAN.search(baris[j]):
                    return i, j
                if POLA_MENGADILI.match(baris[j]) or POLA_PENUTUP.search(baris[j]):
                    return i, j
                if re.match(r"^\s*Menimbang", baris[j]):
                    return i, j
            return i, min(len(baris), i + 400)
    return None, None
def indeks_mengadili(baris):
    """Semua baris MENGADILI, termasuk yang hurufnya direnggangkan."""
    return [i for i, x in enumerate(baris) if POLA_MENGADILI.match(x)]
def batas_putusan(baris):
    """Amar putusan diambil dari MENGADILI terakhir sebelum penutup."""
    penanda = indeks_mengadili(baris)
    if not penanda:
        return None, None
    awal_p, akhir_p = batas_penutup(baris)
    kandidat = [i for i in penanda if awal_p is None or i < awal_p]
    awal = kandidat[-1] if kandidat else penanda[-1]
    akhir = awal_p if awal_p is not None and awal_p > awal else min(len(baris), awal + 120)
    return awal, akhir
def batas_penutup(baris):
    """Penutup dimulai dari kalimat 'Demikian ...' terakhir."""
    mulai = None
    for i, x in enumerate(baris):
        if POLA_PENUTUP.search(x):
            mulai = i
    if mulai is None:
        mulai = max(0, len(baris) - 80)
    return mulai, min(len(baris), mulai + 45)
def teks_penutup(baris):
    """Gabungkan kalimat penutup, buang footer, rapatkan titik gelar ('M.Hum .')."""
    awal, akhir = batas_penutup(baris)
    teks = " ".join(buang_baris_sampah(baris[awal:akhir]))
    teks = re.sub(r"\s+\.", ".", teks)  # 'M.Hum .' -> 'M.Hum.'
    teks = re.sub(r"\b([A-Z])\s+(?=[A-Z]\b)", r"\1", teks)  # 'S H' -> 'SH'
    teks = re.sub(r",\.(?=\s|$)", ",", teks)  # 'MH,.' -> 'MH,'
    teks = re.sub(r"\.(?=Hakim|Panitera|Penuntut|Jaksa)", ". ", teks)  # 'M.Hum.Hakim' -> 'M.Hum. Hakim'
    teks = re.sub(r"([A-Za-z])-([a-z])", r"\1\2", teks)  # 'tang-gal' -> 'tanggal'
    teks = re.sub(r"\bNomor\s*:\s*", "", teks)  # buang label 'Nomor :' yang terpotong footer
    return teks
def teks_item_bernomor(bagian, nomor, frasa=None):
    """Ambil satu amar bernomor ('1.', '1)', '2. ...' di tengah baris, '- ...')."""
    nomor = str(nomor)
    kata_amar = r"(?:menyatakan|menjatuhkan|menetapkan|menghukum|membebankan|memerintahkan)"
    if frasa:
        kunci = frasa.lower()
        for i, x in enumerate(bagian):
            if kunci in x.lower():
                jangkau = range(max(0, i - 5), min(len(bagian), i + 6))
                tepat = [k for k in jangkau if re.match(rf"^\s*{nomor}[.)]\s*(.*)$", bagian[k]) or re.search(rf"(?:^|\s){nomor}[.)]\s*(?:$|{kata_amar})", bagian[k], re.I)]
                garis = [k for k in jangkau if POLA_GARIS_AMAR.match(bagian[k])]
                target = tepat + [k for k in garis if k not in tepat]
                if not target:
                    continue
                k = target[0]
                mulai = min(i, k)
                akhir = len(bagian)
                for j in range(max(i, k) + 1, len(bagian)):
                    if POLA_NOMOR_AMAR.match(bagian[j]):
                        akhir = j
                        break
                else:
                    for j in range(max(i, k) + 1, len(bagian)):
                        if POLA_GARIS_AMAR.match(bagian[j]):
                            akhir = j
                            break
                return " ".join(buang_baris_sampah(bagian[mulai:akhir]))
    for i, x in enumerate(bagian):
        cocok = POLA_NOMOR_AMAR.match(x)
        if cocok and cocok.group(1) == nomor:
            akhir = len(bagian)
            for j in range(i + 1, len(bagian)):
                if awal_amar_baru(bagian[j]):
                    akhir = j
                    break
            return " ".join(buang_baris_sampah(bagian[i:akhir]))
    return None

def ambil_lama_pidana(teks):
    """Ambil durasi penjara sesudah kata 'selama' ('1 tahun dan 6 bulan')."""
    if not teks:
        return None
    khusus = POLA_HUKUMAN_KHUSUS.search(teks)
    if khusus:
        return rapikan_teks(khusus.group(0))
    for awal in re.finditer(r"(?i)\bs(?:elama|leama)\b\s*:?\s*(.{0,300})", teks):
        hasil = _lama_dari_sisa(awal.group(1))
        if hasil:
            return hasil
    return None
def _lama_dari_sisa(sisa):
    """Rangkai token angka + satuan waktu sampai kata penghenti."""
    token = re.findall(r"\(|\)|,|;|:|\d+(?:[.,]\d+)?|[A-Za-zÀ-ž]+", sisa)
    potongan = []
    ada_satuan = False
    i = 0
    while i < len(token):
        kata = token[i]
        rendah = kata.casefold()
        if kata in {",", ";", ":"}:
            break
        if rendah in KATA_BERHENTI_LAMA:
            break
        if rendah == "dan" or rendah == "clan":  # 'clan' = salah ketik OCR untuk 'dan'
            berikut = token[i + 1] if i + 1 < len(token) else ""
            if re.fullmatch(r"\d+(?:[.,]\d+)?", berikut):
                potongan.append(kata)
                i += 1
                continue
            break
        potongan.append(kata)
        if rendah in SATUAN_WAKTU:
            ada_satuan = True
        i += 1
    if not ada_satuan:
        return None
    gabung = " ".join(potongan)
    gabung = re.sub(r"\(\s+", "(", gabung)
    gabung = re.sub(r"\s+\)", ")", gabung)
    return rapikan_teks(gabung)


In [15]:
# ================= BAGIAN 10: Fungsi entitas 1-9 =================
def format_nomor(kecocokan):
    nomor = kecocokan.group(1).strip()
    if set(nomor) == {"-"}:
        return None
    jenis = re.sub(r"\s*/\s*", "/", kecocokan.group(2).strip())
    tahun = kecocokan.group(3).strip()
    pengadilan = kecocokan.group(4)
    if pengadilan:
        pengadilan = re.sub(r"\s+", " ", pengadilan).rstrip(".")
        return f"{nomor}/{jenis}/{tahun}/PN {pengadilan}"
    return f"{nomor}/{jenis}/{tahun}"
def ambil_nomor_putusan(baris):
    for x in baris[:30]:
        if POLA_BARIS_NOMOR.search(x):
            kanan = x.split(":", 1)[1] if ":" in x else x
            cocok = POLA_NOMOR.search(kanan)
            if cocok:
                return format_nomor(cocok)
    for x in baris[:30]:
        cocok = POLA_NOMOR.search(x)
        if cocok:
            return format_nomor(cocok)
    return None
def ambil_satu_nama(baris, i, selesai):
    mentah = POLA_LABEL_NAMA.split(baris[i], maxsplit=1)[-1]
    calon = [mentah]
    if ";" not in mentah:
        for j in range(i + 1, min(selesai, i + 5)):
            berikut = baris[j]
            if not berikut.strip():
                break
            if POLA_LABEL_NAMA.search(berikut) or POLA_LABEL_LAPANGAN.search(berikut):
                break
            calon.append(berikut)
            if ";" in berikut:
                break
    return bersihkan_nama_terdakwa(" ".join(calon))
def ambil_nama_terdakwa(baris, jenis):
    if jenis != "pidana":
        return None, None, None
    selesai = batas_identitas(baris)
    nama = []
    for i, x in enumerate(baris[:selesai]):
        if POLA_LABEL_NAMA.search(x):
            bersih = ambil_satu_nama(baris, i, selesai)
            if bersih and len(bersih) > 3 and bersih not in nama:
                nama.append(bersih)
        if len(nama) >= 3:
            break
    nama = nama[:3]
    while len(nama) < 3:
        nama.append(None)
    return nama[0], nama[1], nama[2]
def ambil_perbuatan(teks):
    pola = re.search(r"(?i)bersalah\s+melakukan\s+(.*)", teks)
    if not pola:
        pola = re.search(r"(?i)\bmelakukan\b\s+(.*)", teks)
    if not pola:  # varian: '... menurut hukum dalam hal perbuatan X ...'
        pola = re.search(r"(?i)terbukti\b.{0,120}?menurut hukum\s+dalam hal\s+(?:perbuatan\s+)?(.*)", teks)
    if not pola:
        return None
    depan = re.split(r"(?i)\s*,?\s*sebagaimana|;|\bkarena\b|\bmelanggar\b|\bdiatur\b|\bdakwaan\b", pola.group(1), maxsplit=1)[0]
    depan = re.sub(r"^(?:[A-ZÀ-Ž][A-ZÀ-Ž.'’\-]*\s+){1,3}(?=[Tt]indak\s+[Pp]idana)", "", depan)
    depan = re.sub(r"(?i)^(?:perbuatan\s*,\s*|tindak\s+pidana\s+|pidana\s+|telah\s+melakukan\s+)+", "", depan)
    depan = rapikan_teks(depan)
    if depan and depan.lower() in {"tindak pidana", "pidana", "perbuatan", "tindak pidana tersebut"}:
        return None  # hanya label generik, bukan nama perbuatannya
    return depan
def ambil_statuta_kuhp(teks):
    pola = re.search(r"(?i)pasal\s+\d+.{0,200}?\s+KUH\s*Pidana|pasal\s+\d+.{0,200}?\s+KUHP", teks)
    return rapikan_teks(pola.group(0)) if pola else None
def ambil_tuntutan_pidana(baris, baris_low, jenis):
    if jenis != "pidana":
        return None
    awal, akhir = batas_tuntutan(baris, baris_low)
    if awal is None:
        return None
    item = teks_item_bernomor(baris[awal:akhir], "1", "menyatakan")
    return ambil_perbuatan(item) if item else None
def ambil_tuntutan_kuhp(baris, baris_low, jenis):
    if jenis != "pidana":
        return None
    awal, akhir = batas_tuntutan(baris, baris_low)
    if awal is None:
        return None
    item = teks_item_bernomor(baris[awal:akhir], "1", "menyatakan")
    return ambil_statuta_kuhp(item) if item else None
def ambil_tuntutan_hukuman(baris, baris_low, jenis):
    if jenis != "pidana":
        return None
    awal, akhir = batas_tuntutan(baris, baris_low)
    if awal is None:
        return None
    item = teks_item_bernomor(baris[awal:akhir], "2", "menjatuhkan")
    return ambil_lama_pidana(item) if item else None
def ambil_putusan_pidana(baris, jenis):
    if jenis != "pidana":
        return None
    awal, akhir = batas_putusan(baris)
    if awal is None:
        return None
    item = teks_item_bernomor(baris[awal:akhir], "1", "menyatakan")
    return ambil_perbuatan(item) if item else None
def ambil_putusan_hukuman(baris, jenis):
    if jenis != "pidana":
        return None
    awal, akhir = batas_putusan(baris)
    if awal is None:
        return None
    item = teks_item_bernomor(baris[awal:akhir], "2", "menjatuhkan")
    return ambil_lama_pidana(item) if item else None
def normalisasi_bulan(nama):
    peta = {"januari":"Januari","februari":"Februari","pemberi":"Februari","pebruari":"Februari","maret":"Maret","april":"April","mei":"Mei","juni":"Juni","juli":"Juli","agustus":"Agustus","september":"September","sepetember":"September","oktober":"Oktober","nopember":"November","november":"November","desember":"Desember"}
    return peta.get(nama.lower(), nama)
def ambil_tanggal_putusan(baris):
    cocok = POLA_TANGGAL.search(teks_penutup(baris))
    if not cocok:
        return None
    return f"{int(cocok.group(2))} {normalisasi_bulan(cocok.group(3))} {cocok.group(4)}"
def cari_panel(penutup):
    """Kembalikan cocok panel hakim: baku, ad-hoc, lalu tanpa label anggota."""
    return (POLA_PANEL_HAKIM.search(penutup) or POLA_PANEL_HAKIM_ADHOC.search(penutup) or POLA_PANEL_TANPA_LABEL.search(penutup))
def ambil_hakim_ketua(baris):
    penutup = teks_penutup(baris)
    panel = cari_panel(penutup)
    if panel:
        return rapikan_nama(panel.group("ketua"))
    for cocok in re.finditer(r"(?i:\boleh)(?:\s+kami)?\s*[,:;]?\s*(" + TEKS_NAMA + r")", penutup):
        sesudah = penutup[cocok.end():cocok.end() + 250]
        if re.search(r"(?i)\bHakim\b(?!\s+(?:Anggota|Ketua|Majelis|-|,))", sesudah) and "masing-masing" not in sesudah.lower():
            return rapikan_nama(cocok.group(1))
    return None
def ambil_hakim_anggota(baris):
    panel = cari_panel(teks_penutup(baris))
    if not panel:
        return None, None
    daftar = pecah_nama_bergelar(panel.group("anggota"))
    if len(daftar) >= 2:
        return daftar[0], daftar[1]
    if len(daftar) == 1:
        return daftar[0], None
    return None, None
POLA_PANITERA_APOSISI = re.compile(fr"(?s)(?:[,;:]|\btersebut\b)\s*(?P<nama>{TEKS_NAMA})[ \t]*,?[ \t]*Panitera\s+Pengganti")
def cocok_terdekat(pola, teks):
    """Pilih cocok dengan rentang terpendek (nama paling dekat dengan jabatannya)."""
    semua = list(pola.finditer(teks))
    if not semua:
        return None
    return min(semua, key=lambda m: m.end() - m.start())
def ambil_panitera(baris):
    penutup = teks_penutup(baris)
    cocok = cocok_terdekat(POLA_PANITERA, penutup)
    if not cocok:  # cadangan: aposisi ', NAMA, Panitera Pengganti' tanpa kata 'dibantu'
        cocok = cocok_terdekat(POLA_PANITERA_APOSISI, penutup)
    if not cocok:
        return None
    return rapikan_nama(cocok.group("nama"))
def ambil_penuntut_umum(baris):
    penutup = teks_penutup(baris)
    cocok = cocok_terdekat(POLA_JPU, penutup)
    if cocok:
        return rapikan_teks(cocok.group("nama"))
    cocok = cocok_terdekat(POLA_JPU_POLOS, penutup)  # cadangan: nama tanpa gelar
    if cocok:
        nama = rapikan_nama(cocok.group("nama"))
        if nama and not re.search(r"(?i)\b(terdakwa|pemohon|advokat|kuasa|hakim|panitera|para|yang)\b", nama):
            return nama
    return None



In [16]:
# ================= BAGIAN 10: Orkestrator ekstrak() =================
def tulis_keluaran_txt(lintasan, hasil):
    """Tulis satu entitas per baris memakai label yang mudah dibaca."""
    with open(lintasan, "w", encoding="utf-8") as berkas:
        berkas.write("=== ENTITAS TEREKSTRAKSI ===\n")
        berkas.write(f"File Sumber   : {hasil['file_txt']}\n")
        berkas.write(f"Jenis Dokumen : {hasil['jenis_dokumen']}\n")
        berkas.write(f"Status        : {hasil['status_lengkap']} ({hasil['total_entitas_terisi']}/{hasil['entitas_berlaku']} berlaku)\n")
        berkas.write("-" * 60 + "\n")
        for kunci in KOLOM_ENTITAS:
            berkas.write(f"{LABEL_CETAK[kunci]:17s} : {hasil[kunci]}\n")
        berkas.write("-" * 60 + "\n")
        berkas.write("Catatan Kosong:\n")
        berkas.write(hasil["catatan_kosong"] + "\n")
def ekstrak(lintasan_txt: Path):
    """Ekstrak 15 entitas dan keterangan dari satu file."""
    alasan = {}
    try:
        baris, baris_low = baca_baris(lintasan_txt)
        jenis = nama_jenis_dokumen(lintasan_txt.name)
        berlaku = himpunan_berlaku(jenis)
        hasil = {kunci: TANDA_KOSONG for kunci in KOLOM_ENTITAS}
        if "nomor_putusan" in berlaku:
            nilai = ambil_nomor_putusan(baris)
            if nilai:
                hasil["nomor_putusan"] = nilai
            else:
                alasan["nomor_putusan"] = "nomor tidak terbaca pada judul dokumen"
        for kunci, nilai in zip(["nama_terdakwa_1", "nama_terdakwa_2", "nama_terdakwa_3"], ambil_nama_terdakwa(baris, jenis)):
            if kunci not in berlaku:
                alasan[kunci] = f"tidak berlaku untuk dokumen {jenis}"
            elif nilai:
                hasil[kunci] = nilai
            elif kunci == "nama_terdakwa_1":
                alasan[kunci] = "label Nama lengkap/terdakwa tidak ditemukan di blok identitas"
            else:
                alasan[kunci] = "dokumen ini tidak mempunyai terdakwa kedua/ketiga"
        for kunci, fungsi, pesan in [
            ("tuntutan_pidana", lambda: ambil_tuntutan_pidana(baris, baris_low, jenis), "amar Tuntutan nomor 1 tidak memuat perbuatan yang dapat dipisahkan"),
            ("tuntutan_kuhp", lambda: ambil_tuntutan_kuhp(baris, baris_low, jenis), "amar Tuntutan nomor 1 tidak memuat Pasal KUHP"),
            ("tuntutan_hukuman", lambda: ambil_tuntutan_hukuman(baris, baris_low, jenis), "amar Tuntutan nomor 2 tidak memuat durasi sesudah 'selama'"),
            ("putusan_pidana", lambda: ambil_putusan_pidana(baris, jenis), "amar MENGADILI nomor 1 tidak memuat perbuatan yang dapat dipisahkan"),
            ("putusan_hukuman", lambda: ambil_putusan_hukuman(baris, jenis), "amar MENGADILI nomor 2 tidak memuat durasi sesudah 'selama'"),
        ]:
            if kunci not in berlaku:
                alasan[kunci] = f"tidak berlaku untuk dokumen {jenis}"
                continue
            nilai = fungsi()
            if nilai:
                hasil[kunci] = nilai
            else:
                alasan[kunci] = pesan
        nilai = ambil_tanggal_putusan(baris)
        if nilai:
            hasil["tanggal_putusan"] = nilai
        else:
            alasan["tanggal_putusan"] = "tanggal 'pada hari ..., tanggal ...' tidak ditemukan di penutup"
        nilai = ambil_hakim_ketua(baris)
        if nilai:
            hasil["hakim_ketua"] = nilai
        else:
            alasan["hakim_ketua"] = "nama sebelum 'Hakim Ketua' tidak ditemukan di penutup"
        pertama, kedua = ambil_hakim_anggota(baris)
        if pertama:
            hasil["hakim_anggota_1"] = pertama
        else:
            alasan["hakim_anggota_1"] = "daftar 'masing-masing sebagai Hakim Anggota' tidak ditemukan (misalnya hakim tunggal)"
        if kedua:
            hasil["hakim_anggota_2"] = kedua
        else:
            alasan["hakim_anggota_2"] = "anggota kedua tidak ditemukan (misalnya hakim tunggal)"
        nilai = ambil_panitera(baris)
        if nilai:
            hasil["panitera"] = nilai
        else:
            alasan["panitera"] = "pola 'dibantu oleh ... Panitera Pengganti' tidak ditemukan"
        if "penuntut_umum" not in berlaku:
            alasan["penuntut_umum"] = f"tidak berlaku untuk dokumen {jenis}"
        else:
            nilai = ambil_penuntut_umum(baris)
            if nilai:
                hasil["penuntut_umum"] = nilai
            else:
                alasan["penuntut_umum"] = "nama sebelum 'Penuntut Umum' tidak ditemukan di penutup"
        terisi = sum(1 for kunci in berlaku if hasil.get(kunci) not in (TANDA_KOSONG, None, ""))
        if terisi == len(berlaku):
            status = "lengkap"
        elif terisi == 0:
            status = "kosong"
        else:
            status = "sebagian"
        hilang = [kunci for kunci in KOLOM_ENTITAS if hasil.get(kunci) == TANDA_KOSONG]
        catatan = "; ".join(f"{kunci}: {alasan.get(kunci, 'tidak ditemukan')}" for kunci in hilang)
        return {**hasil, "_ok": True, "_error": None, "file_txt": lintasan_txt.name, "jenis_dokumen": jenis, "entitas_berlaku": len(berlaku), "total_entitas_terisi": sum(1 for kunci in KOLOM_ENTITAS if hasil.get(kunci) not in (TANDA_KOSONG, None, "")), "status_lengkap": status, "catatan_kosong": catatan or "semua entitas yang berlaku terisi"}
    except Exception as salah:
        kosong = {kunci: TANDA_KOSONG for kunci in KOLOM_ENTITAS}
        return {**kosong, "_ok": False, "_error": str(salah), "file_txt": lintasan_txt.name, "jenis_dokumen": "tidak-diketahui", "entitas_berlaku": 0, "total_entitas_terisi": 0, "status_lengkap": "gagal", "catatan_kosong": f"ERROR: {salah}"}


In [17]:
# ================= BAGIAN 10: Demo 1 file + print proses scraping =================
contoh = INPUT_TXT / CONTOH_DEMO
if not contoh.exists():
    kandidat = sorted([x for x in INPUT_TXT.glob("T-*.txt") if "_Pid.B_" in x.name or "_Pid.Sus" in x.name])
    contoh = kandidat[0]
print(">> Proses scraping entitas")
print(f">> Membaca: {contoh.name}")
hasil = ekstrak(contoh)
print(f">> Jenis dokumen: {hasil['jenis_dokumen']} | Terisi: {hasil['total_entitas_terisi']}/{hasil['entitas_berlaku']} | Status: {hasil['status_lengkap']}")
print(">> Hasil tiap entitas dan aturan yang dipakai:")
for kunci in KOLOM_ENTITAS:
    tanda = "✓" if hasil[kunci] != TANDA_KOSONG else "✗"
    print(f"  {tanda} {LABEL_CETAK[kunci]:17s} = {hasil[kunci]}")
    print(f"    aturan: {JEJAK_ATURAN[kunci]}")
print(">> Alasan nilai kosong:")
print(f"    {hasil['catatan_kosong']}")
tulis_keluaran_txt(OUTPUT_CLEAN / f"O-{contoh.name}", hasil)
print(f">> Ditulis: OUTPUT_CLEAN/O-{contoh.name}")


>> Proses scraping entitas
>> Membaca: T-1016_Pid.B_2026_PN Sby.txt
>> Jenis dokumen: pidana | Terisi: 12/15 | Status: sebagian
>> Hasil tiap entitas dan aturan yang dipakai:
  ✓ Nomor Putusan     = 1016/Pid.B/2026/PN Sby
    aturan: baris judul 'Nomor :' + POLA_NOMOR
  ✓ Nama Terdakwa 1   = Vicky Dubes Borty Bin M Irfan Wijaya
    aturan: label Nama di blok identitas pidana
  ✗ Nama Terdakwa 2   = -
    aturan: label Nama kedua di blok identitas pidana
  ✗ Nama Terdakwa 3   = -
    aturan: label Nama ketiga di blok identitas pidana
  ✓ Tuntutan Pidana   = yang secara melawan hukum memiliki suatu barang yang sebagian atau seluruhnya milik orang lain, yang ada dalam kekuasaannya bukan
    aturan: amar 1 Tuntutan + frasa sesudah 'bersalah melakukan'
  ✗ Tuntutan KUHP     = -
    aturan: amar 1 Tuntutan + pola 'Pasal ... KUHP'
  ✓ Tuntutan Hukuman  = 1 (satu) tahun 3 (tiga) bulan
    aturan: amar 2 Tuntutan + durasi sesudah 'selama'
  ✓ Putusan Pidana    = Penggelepan
    aturan: amar 1 M

In [18]:
# ================= BAGIAN 10: Batch semua file menjadi TXT + CSV =================
semua = sorted(INPUT_TXT.glob("T-*.txt"))
print(f">> Memproses {len(semua)} file dari CLEAN_TXT/ ...")
baris_csv = []
berhasil = 0
gagal = 0
for berkas in semua:
    hasil = ekstrak(berkas)
    if hasil.get("_ok"):
        berhasil += 1
    else:
        gagal += 1
    tulis_keluaran_txt(OUTPUT_CLEAN / f"O-{berkas.name}", hasil)
    baris = []
    for kunci in KOLOM_CSV:
        nilai = hasil.get(kunci, TANDA_KOSONG)
        baris.append(str(nilai if nilai is not None else TANDA_KOSONG))
    baris_csv.append(baris)
jalur_csv = OUTPUT_CLEAN / "entitas_258.csv"
with open(jalur_csv, "w", newline="", encoding="utf-8") as csv_file:
    tulis = csv.writer(csv_file)
    tulis.writerow(KOLOM_CSV)
    tulis.writerows(baris_csv)
print(">> Rekap batch:")
print(f"  Total file : {len(semua)}")
print(f"  Berhasil   : {berhasil}")
print(f"  Gagal      : {gagal}")
print(f">> CSV       : {jalur_csv}")
print(f">> TXT       : {OUTPUT_CLEAN} (O-*.txt)")


>> Memproses 258 file dari CLEAN_TXT/ ...
>> Rekap batch:
  Total file : 258
  Berhasil   : 258
  Gagal      : 0
>> CSV       : OUTPUT_CLEAN/entitas_258.csv
>> TXT       : OUTPUT_CLEAN (O-*.txt)


In [19]:
# ================= BAGIAN 10: Tampilkan 5 data teratas dengan print() =================
jalur_csv = OUTPUT_CLEAN / "entitas_258.csv"
if not jalur_csv.exists():
    print("CSV belum ada. Jalankan cell batch terlebih dahulu.")
else:
    with open(jalur_csv, encoding="utf-8") as csv_file:
        semua_baris = list(csv.DictReader(csv_file))
    print(f">> Menampilkan {BATAS_PREVIEW} data teratas dari {jalur_csv}")
    print("=" * 100)
    for nomor, baris in enumerate(semua_baris[:BATAS_PREVIEW], 1):
        print(f"\n[{nomor}] {baris['file_txt']} | {baris['jenis_dokumen']} | {baris['total_entitas_terisi']}/{baris['entitas_berlaku']} | {baris['status_lengkap']}")
        print("-" * 100)
        for kunci in KOLOM_ENTITAS:
            print(f"  {LABEL_CETAK[kunci]:17s} : {baris[kunci]}")
        print(f"  Catatan     : {baris['catatan_kosong']}")
    print("\n" + "=" * 100)
    print(">> Rekap status:", dict(Counter(x["status_lengkap"] for x in semua_baris)))
    print(">> Rekap jenis :", dict(Counter(x["jenis_dokumen"] for x in semua_baris)))


>> Menampilkan 5 data teratas dari OUTPUT_CLEAN/entitas_258.csv

[1] T-1002_Pid.Sus-LH_2026_PN Sby.txt | pidana | 12/15 | sebagian
----------------------------------------------------------------------------------------------------
  Nomor Putusan     : 1002/Pid.Sus-LH/2026/PN Sby
  Nama Terdakwa 1   : HENDRIK SETIAWAN BIN (alm) ABDUL HALIM
  Nama Terdakwa 2   : -
  Nama Terdakwa 3   : -
  Tuntutan Pidana   : menyalagunakan Pengangkutan dan/atau Niaga Bahan Bakar Minyak, Bahan Bakar Gas, dan/atau liquefied petroleum gas yang disbusidi dan/atau penyediaan dan pendistribusiannya diberikan penugasan pemerintah
  Tuntutan KUHP     : -
  Tuntutan Hukuman  : 1 (satu) Tahun dan 6 (enam) bulan
  Putusan Pidana    : menyalahgunakan niaga bahan bakar minyak yang disubsidi dan atau penyediaan dan pendistribusiannya diberikan penugasan pemerintah
  Putusan Hukuman   : 1 (satu) tahun dan 2 (dua) bulan
  Tanggal Putusan   : 31 Agustus 2026
  Hakim Ketua       : Alex Adam Faisal, S.H., M.H.
  Hakim A

In [20]:
# ================= BAGIAN 10: Cakupan tiap entitas =================
jalur_csv = OUTPUT_CLEAN / "entitas_258.csv"
if not jalur_csv.exists():
    print("CSV belum ada. Jalankan cell batch terlebih dahulu.")
else:
    with open(jalur_csv, encoding="utf-8") as csv_file:
        semua_baris = list(csv.DictReader(csv_file))
    print(">> Cakupan entitas (terisi dibagi jumlah berlaku)")
    print("-" * 60)
    for kunci in KOLOM_ENTITAS:
        berlaku = sum(1 for x in semua_baris if kunci in himpunan_berlaku(x["jenis_dokumen"]))
        isi = sum(1 for x in semua_baris if x.get(kunci) not in (TANDA_KOSONG, None, ""))
        print(f"  {kunci:20s} : {isi:3d} / {berlaku:3d}")
    print("-" * 60)


>> Cakupan entitas (terisi dibagi jumlah berlaku)
------------------------------------------------------------
  nomor_putusan        : 257 / 258
  nama_terdakwa_1      :  99 /  99
  nama_terdakwa_2      :  14 /  99
  nama_terdakwa_3      :   1 /  99
  tuntutan_pidana      :  88 /  99
  tuntutan_kuhp        :  26 /  99
  tuntutan_hukuman     :  88 /  99
  putusan_pidana       :  98 /  99
  putusan_hukuman      :  99 /  99
  tanggal_putusan      : 258 / 258
  hakim_ketua          : 256 / 258
  hakim_anggota_1      : 155 / 258
  hakim_anggota_2      : 155 / 258
  panitera             : 258 / 258
  penuntut_umum        :  98 /  99
------------------------------------------------------------
